In [0]:
# CÉLULA 1: Instalação (rode isso e reinicie o Python)
%pip install pyettj lxml plotly exchange_calendars
# dbutils.library.restartPython()

In [0]:
import sys
import numpy as np
import pandas as pd
import pyettj
import plotly.graph_objects as go
from zipfile import ZipFile
from lxml import etree
from datetime import date, timedelta
from pathlib import Path

# 1. CONFIGURAÇÃO DE CAMINHOS E DATAS
project_root = Path("/Workspace/Users/luiz.henrique.felipe.rocha@gmail.com/LuizRocha_Databricks_OptionsPrice/ibov-barrier-option-mvp")
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from ibov_barrier.di_curve.curve import DiscountCurve

# Data alvo (o script vai tentar essa data, e se a ANBIMA falhar, volta 1 dia)
DATA_ALVO = date(2026, 10, 27) 

# ==============================================================================
# 2. FUNÇÕES DE EXTRAÇÃO COM TRATAMENTO DE ERROS
# ==============================================================================

def extrair_curva_b3(valuation_date: date) -> pd.DataFrame:
    """Extrai os dados do SPRD da B3 com parsing seguro."""
    sprd_zip = project_root / "data" / valuation_date.isoformat() / f"SPRD{valuation_date:%y%m%d}.zip"
    
    # Verifica se o arquivo existe antes de tentar ler
    if not sprd_zip.exists():
        raise FileNotFoundError(f"❌ Arquivo não encontrado: {sprd_zip}")

    rows = []
    with ZipFile(sprd_zip) as archive:
        member = max([m for m in archive.infolist() if m.filename.lower().endswith(".xml")], key=lambda m: m.date_time)
        with archive.open(member) as f:
            for _, el in etree.iterparse(f, events=("end",), tag="{*}PricRpt", huge_tree=True):
                ticker = el.findtext("{*}SctyId/{*}TckrSymb", default="")
                if ticker.startswith("DI1"):
                    # Correção do Bug: Usar .// para buscar em tags aninhadas
                    rate = el.findtext(".//{*}AdjstdQtTax", default="")
                    if rate:
                        rows.append({"ticker": ticker, "rate": float(rate)/100})
                el.clear()

    # Verifica se encontrou dados antes de criar o DataFrame
    if not rows:
        raise ValueError(f"❌ Nenhum contrato DI1 encontrado no arquivo {sprd_zip.name}. Verifique se os dados estão corretos.")

    df = pd.DataFrame(rows)
    MONTH_CODES = {"F":1, "G":2, "H":3, "J":4, "K":5, "M":6, "N":7, "Q":8, "U":9, "V":10, "X":11, "Z":12}
    
    def parse_maturity(t):
        return pd.Timestamp(year=2000 + int(t[-2:]), month=MONTH_CODES[t[-3]], day=1)

    df['maturity'] = df['ticker'].apply(parse_maturity)
    df = df[df['maturity'].dt.year <= 2035].sort_values('maturity').drop_duplicates('ticker')
    
    df['prazo_anos'] = (df['maturity'] - pd.Timestamp(valuation_date)).dt.days / 365.25
    df['tenor_bd'] = df['prazo_anos'] * 252
    
    return df[['ticker', 'tenor_bd', 'prazo_anos', 'rate']]


def extrair_curvas_anbima(valuation_date: date) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Extrai as curvas da ANBIMA, com fallback para o dia anterior se for feriado."""
    data_loop = valuation_date
    
    for i in range(5): # Tenta hoje, ontem, anteontem...
        data_str = data_loop.strftime("%Y-%m-%d")
        try:
            print(f"Buscando ANBIMA para {data_str}...")
            df_pre = pyettj.get_ettj(data_str, curva="PRE")
            df_dic = pyettj.get_ettj(data_str, curva="DIC")
            
            # Se baixou e tem dados, quebra o loop e formata
            if df_pre is not None and not df_pre.empty and df_dic is not None and not df_dic.empty:
                
                # Formata PRE
                df_pre = df_pre.rename(columns={"dias_uteis": "tenor_bd", "taxa": "taxa_pre"})
                df_pre = df_pre[df_pre["tenor_bd"] > 21].copy()
                df_pre["taxa_pre"] = df_pre["taxa_pre"] * 100
                df_pre["prazo_anos"] = df_pre["tenor_bd"] / 252
                
                # Formata DIC
                df_dic = df_dic.rename(columns={"dias_uteis": "tenor_bd", "taxa": "taxa_dic"})
                df_dic = df_dic[df_dic["tenor_bd"] > 21].copy()
                df_dic["taxa_dic"] = df_dic["taxa_dic"] * 100
                df_dic["prazo_anos"] = df_dic["tenor_bd"] / 252
                
                print(f"✅ ANBIMA carregada com sucesso para {data_str}!")
                return df_pre, df_dic
                
        except Exception as e:
            pass # Se falhar, tenta o dia anterior
            
        # Volta 1 dia
        data_loop = data_loop - timedelta(days=1)
        
    raise ValueError("❌ Não foi possível baixar dados da ANBIMA nos últimos 5 dias. Verifique a conexão ou se os dados existem.")

# ==============================================================================
# 3. FUNÇÕES DE CÁLCULO
# ==============================================================================

def calcular_curva_forward(df_b3: pd.DataFrame) -> pd.DataFrame:
    curva = DiscountCurve.from_di1(df_b3['tenor_bd'].values, df_b3['rate'].values)
    prazos_grid = np.linspace(21, df_b3['tenor_bd'].max(), 100)
    forwards = []
    
    for t in prazos_grid:
        dt = 1.0
        if t + dt > df_b3['tenor_bd'].max(): break
        df_t = curva.df(t)
        df_t_dt = curva.df(t + dt)
        fwd = (df_t / df_t_dt) ** (252 / dt) - 1
        forwards.append({"tenor_bd": t, "prazo_anos": t/252, "forward_rate": fwd * 100})
    return pd.DataFrame(forwards)

def calcular_inflacao_implicita(df_pre: pd.DataFrame, df_dic: pd.DataFrame) -> pd.DataFrame:
    prazos = df_pre['prazo_anos'].values
    taxas_pre = df_pre['taxa_pre'].values / 100
    taxas_dic = np.interp(prazos, df_dic['prazo_anos'], df_dic['taxa_dic']) / 100
    inflacao = ((1 + taxas_pre) / (1 + taxas_dic)) - 1
    return pd.DataFrame({"prazo_anos": prazos, "inflacao_implicita": inflacao * 100})

# ==============================================================================
# 4. FUNÇÕES DE PLOTAGEM
# ==============================================================================

def plotar_curva_di(df_b3):
    fig = go.Figure(go.Scatter(x=df_b3["prazo_anos"], y=df_b3["rate"]*100, mode='lines+markers', 
                               name='Curva DI', line=dict(color='#1f77b4', width=2),
                               hovertemplate="<b>%{text}</b><br>Prazo: %{x:.2f} anos<br>Taxa: %{y:.2f}%<extra></extra>", text=df_b3["ticker"]))
    fig.update_layout(title="1. Curva DI Spot (B3)", xaxis_title="Prazo (Anos)", yaxis_title="Taxa (% a.a.)", hovermode="x unified", plot_bgcolor="white", height=500)
    fig.show()

def plotar_curvas_anbima(df_pre, df_dic):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_pre["prazo_anos"], y=df_pre["taxa_pre"], mode='lines', name='PRE (Nominal)', line=dict(color='#1f77b4', width=2)))
    fig.add_trace(go.Scatter(x=df_dic["prazo_anos"], y=df_dic["taxa_dic"], mode='lines', name='DIC (Real)', line=dict(color='#ff7f0e', width=2)))
    fig.update_layout(title="2. Curvas ANBIMA (PRE vs DIC)", xaxis_title="Prazo (Anos)", yaxis_title="Taxa (% a.a.)", hovermode="x unified", plot_bgcolor="white", height=500)
    fig.show()

def plotar_curva_forward(df_fwd):
    fig = go.Figure(go.Scatter(x=df_fwd["prazo_anos"], y=df_fwd["forward_rate"], mode='lines', name='Forward', line=dict(color='#2ca02c', width=2, dash='dash')))
    fig.update_layout(title="3. Curva Forward", xaxis_title="Prazo (Anos)", yaxis_title="Taxa Forward (% a.a.)", hovermode="x unified", plot_bgcolor="white", height=500)
    fig.show()

def plotar_inflacao_implicita(df_inf):
    fig = go.Figure(go.Scatter(x=df_inf["prazo_anos"], y=df_inf["inflacao_implicita"], mode='lines+markers', name='Inflação Implícita', line=dict(color='#d62728', width=2), fill='tozeroy', fillcolor='rgba(214,39,40,0.2)'))
    fig.update_layout(title="4. Inflação Implícita (Spread PRE vs DIC)", xaxis_title="Prazo (Anos)", yaxis_title="Inflação (% a.a.)", hovermode="x unified", plot_bgcolor="white", height=500)
    fig.show()

# ==============================================================================
# 5. EXECUÇÃO
# ==============================================================================
if __name__ == "__main__":
    print(f"Iniciando processamento para a data alvo: {DATA_ALVO}")
    
    # 1. B3 (Se falhar, avisa claramente)
    try:
        df_b3 = extrair_curva_b3(DATA_ALVO)
        plotar_curva_di(df_b3)
        df_forward = calcular_curva_forward(df_b3)
        plotar_curva_forward(df_forward)
    except Exception as e:
        print(f"Erro na B3: {e}")

    # 2. ANBIMA (Com fallback de data)
    try:
        df_pre, df_dic = extrair_curvas_anbima(DATA_ALVO)
        plotar_curvas_anbima(df_pre, df_dic)
        df_inflacao = calcular_inflacao_implicita(df_pre, df_dic)
        plotar_inflacao_implicita(df_inflacao)
    except Exception as e:
        print(f"Erro na ANBIMA: {e}")